## BLOCKAGE SYNTHETIC DATA GENRATION FOR FAULT TRAINING

In [58]:
# Import necessary libraries
import numpy as np
import pandas as pd
from pathlib import Path
import os

# Reproducibility
np.random.seed(42)


In [59]:
# Blockage configuration
pipe_length   = 50.0
pipe_diameter = 0.2
pipe_radius   = 0.1

nodes_per_timestep = 146
timesteps          = 700

upstream_decay   = 2 * pipe_diameter
downstream_decay = 5 * pipe_diameter

baseline = {
    'pressure':             19657.0,
    'pressure-coefficient': 32600.0,
    'density':              2700.0,
    'velocity-magnitude':   2.479,
    'x-velocity':           2.478,
    'y-velocity':           0.0017,
    'z-velocity':          -0.0013,
    'turb-kinetic-energy':  0.0363,
    'turb-diss-rate':       24.4,
    'wall-shear':           10.4,
    'tailings-vof':         0.400,
}

noise = {
    'pressure':             500.0,
    'pressure-coefficient': 800.0,
    'density':              0.0,
    'velocity-magnitude':   0.05,
    'x-velocity':           0.05,
    'y-velocity':           0.005,
    'z-velocity':           0.005,
    'turb-kinetic-energy':  0.002,
    'turb-diss-rate':       1.5,
    'wall-shear':           0.5,
    'tailings-vof':         0.005,
}

blockage_locations = [8.0, 16.0, 25.0, 34.0, 42.0]

blockage_severities = {
    '25': 0.25,
    '50': 0.55,
    '75': 0.90,
}

label_blockage= 2

print("Blockage config loaded.")
print(f"Locations  : {blockage_locations}")
print(f"Severities : {list(blockage_severities.keys())}")
print(f"Total cases: {len(blockage_locations) * len(blockage_severities)}")
print(f"Rows/case  : {nodes_per_timestep * timesteps:,}")

Blockage config loaded.
Locations  : [8.0, 16.0, 25.0, 34.0, 42.0]
Severities : ['25', '50', '75']
Total cases: 15
Rows/case  : 102,200


In [60]:
# Output directory
Output_dir = Path(os.path.join(os.path.dirname(os.getcwd()), "data", "synthetic"))
Output_dir.mkdir(parents=True, exist_ok=True)

print("Generating synthetic dataset for leak and blockage scenarios...")
print("Total samples per scenario: ", nodes_per_timestep * timesteps)
print("Total scenarios: ", len(blockage_locations) * len(blockage_severities))  # +1 for normal
print("Total blockage locations: ", len(blockage_locations))
print("Total severity levels: ", len(blockage_severities))
print(Output_dir)

Generating synthetic dataset for leak and blockage scenarios...
Total samples per scenario:  102200
Total scenarios:  15
Total blockage locations:  5
Total severity levels:  3
/home/darlenewendie/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/synthetic


In [61]:
# Generate node coordinates along the pipe length with random radial positions
def generate_node_coordinates(n_nodes=nodes_per_timestep):
    np.random.seed(42)
    node_numbers = np.arange(1, n_nodes + 1)
    x_coords = np.linspace(0, pipe_length, n_nodes)
    angles   = np.linspace(0, 2 * np.pi, n_nodes)
    radii    = np.random.uniform(0, pipe_radius, n_nodes)
    y_coords = radii * np.cos(angles)
    z_coords = radii * np.sin(angles)
    return node_numbers, x_coords, y_coords, z_coords

def generate_normal_pressure_profile(x_coords):
    P_inlet  = 40000.0
    P_outlet = 0.0
    return P_inlet - (P_inlet - P_outlet) * (x_coords / pipe_length)

node_numbers, x_coords, y_coords, z_coords = generate_node_coordinates()
print(f"Nodes: {len(node_numbers)}")
print(f"X range: {x_coords.min():.1f} → {x_coords.max():.1f} m")

Nodes: 146
X range: 0.0 → 50.0 m


In [62]:
# Generate presure disturbance for a given blockage severity and location
def apply_blockage_pressure_disturbance(pressure, x_coords,
                                         block_x, effect_factor):
    pressure = pressure.copy()

    max_upstream_rise   = 12000.0 * effect_factor
    max_downstream_drop = 10000.0 * effect_factor

    for i, x in enumerate(x_coords):
        distance = x - block_x

        if distance < 0:
            # UPSTREAM — pressure RISES (back pressure)
            upstream_dist = abs(distance)
            rise = max_upstream_rise * np.exp(
                -upstream_dist / upstream_decay)
            pressure[i] += rise

        elif abs(distance) < 0.1:
            # AT BLOCKAGE CENTER — sharp spike
            pressure[i] += max_upstream_rise * 0.8

        else:
            # DOWNSTREAM — significantly lower pressure
            drop = max_downstream_drop * np.exp(
                -distance / downstream_decay)
            pressure[i] -= drop

    return pressure

In [63]:
# Generate velocity disturbance for a given blockage severity and location
def apply_blockage_velocity_disturbance(velocity_mag, x_velocity,
                                         y_velocity, z_velocity,
                                         x_coords, block_x, effect_factor):
    velocity_mag = velocity_mag.copy()
    x_velocity   = x_velocity.copy()
    y_velocity   = y_velocity.copy()
    z_velocity   = z_velocity.copy()

    max_vel_drop          = 1.5 * effect_factor
    upstream_reduction    = 0.3 * effect_factor

    for i, x in enumerate(x_coords):
        distance = x - block_x

        if distance < 0:
            # UPSTREAM — slight velocity decrease
            upstream_dist = abs(distance)
            reduction = upstream_reduction * np.exp(
                -upstream_dist / upstream_decay)
            velocity_mag[i] = max(0, velocity_mag[i] - reduction)
            x_velocity[i]   = max(0, x_velocity[i]   - reduction)

        elif abs(distance) < 0.2:
            # AT BLOCKAGE — sharp velocity drop
            local_drop = max_vel_drop * np.exp(
                -abs(distance) / pipe_diameter)
            velocity_mag[i] = max(0, velocity_mag[i] - local_drop)
            x_velocity[i]   = max(0, x_velocity[i]   - local_drop)
            # No significant radial components for blockage

        else:
            # DOWNSTREAM — sustained reduction
            downstream_drop = max_vel_drop * 0.7 * np.exp(
                -distance / downstream_decay)
            velocity_mag[i] = max(0, velocity_mag[i] - downstream_drop)
            x_velocity[i]   = max(0, x_velocity[i]   - downstream_drop)

    return velocity_mag, x_velocity, y_velocity, z_velocity

In [64]:
# Blockage turbulence disturbance
def apply_blockage_turbulence_disturbance(tke, tdr, x_coords,
                                           block_x, effect_factor):
    tke = tke.copy()
    tdr = tdr.copy()

    max_tke_increase = 0.012 * effect_factor
    max_tdr_increase = 50.0  * effect_factor

    for i, x in enumerate(x_coords):
        distance = x - block_x

        if abs(distance) < 0.5:
            # SYMMETRIC around blockage
            effect = np.exp(-abs(distance) / (2 * pipe_diameter))
            tke[i] += max_tke_increase * effect
            tdr[i] += max_tdr_increase * effect

        elif distance > 0:
            # DOWNSTREAM — recovering
            downstream_effect = 0.5 * np.exp(
                -distance / downstream_decay)
            tke[i] += max_tke_increase * downstream_effect
            tdr[i] += max_tdr_increase * downstream_effect

    return tke, tdr

In [65]:
# Blockage tailings with vof disturbance
def apply_blockage_tailings_disturbance(tailings_vof, x_coords,
                                         block_x, effect_factor):
    tailings_vof = tailings_vof.copy()

    max_upstream_increase = 0.06 * effect_factor
    max_at_block_increase = 0.08 * effect_factor
    max_downstream_drop   = 0.07 * effect_factor

    for i, x in enumerate(x_coords):
        distance = x - block_x

        if distance < 0:
            # UPSTREAM — solids accumulate
            upstream_dist = abs(distance)
            increase = max_upstream_increase * np.exp(
                -upstream_dist / upstream_decay)
            tailings_vof[i] = min(0.508, tailings_vof[i] + increase)

        elif abs(distance) < 0.2:
            # AT BLOCKAGE — maximum solids
            tailings_vof[i] = min(0.508,
                tailings_vof[i] + max_at_block_increase)

        else:
            # DOWNSTREAM — solids starved
            downstream_drop = max_downstream_drop * np.exp(
                -distance / downstream_decay)
            tailings_vof[i] = max(0.163, tailings_vof[i] - downstream_drop)

    return tailings_vof

In [66]:
# Blockage wall shear disturbance
def apply_blockage_wall_shear_disturbance(wall_shear, x_coords,
                                           block_x, effect_factor):
    wall_shear = wall_shear.copy()

    max_increase = 20.0 * effect_factor

    for i, x in enumerate(x_coords):
        distance = x - block_x

        if abs(distance) < 0.3:
            # AT BLOCKAGE — high wall shear
            local_effect = max_increase * np.exp(
                -abs(distance) / pipe_diameter)
            wall_shear[i] += local_effect

    return wall_shear

In [67]:
# Single blockage timestamp generator
def generate_blockage_timestep(timestep, block_x, effect_factor,
                                node_numbers, x_coords,
                                y_coords, z_coords):
    n = len(x_coords)

    #  Normal baseline
    pressure   = generate_normal_pressure_profile(x_coords)
    pressure  += np.random.normal(0, noise['pressure'], n)

    density    = np.full(n, baseline['density'])

    vel_mag    = np.full(n, baseline['velocity-magnitude']) + \
                 np.random.normal(0, noise['velocity-magnitude'], n)
    x_vel      = np.full(n, baseline['x-velocity']) + \
                 np.random.normal(0, noise['x-velocity'], n)
    y_vel      = np.full(n, baseline['y-velocity']) + \
                 np.random.normal(0, noise['y-velocity'], n)
    z_vel      = np.full(n, baseline['z-velocity']) + \
                 np.random.normal(0, noise['z-velocity'], n)

    tke        = np.full(n, baseline['turb-kinetic-energy']) + \
                 np.random.normal(0, noise['turb-kinetic-energy'], n)
    tdr        = np.full(n, baseline['turb-diss-rate']) + \
                 np.random.normal(0, noise['turb-diss-rate'], n)
    wall_shear = np.full(n, baseline['wall-shear']) + \
                 np.random.normal(0, noise['wall-shear'], n)
    tailings   = np.full(n, baseline['tailings-vof']) + \
                 np.random.normal(0, noise['tailings-vof'], n)

    # Apply blockage disturbances
    pressure = apply_blockage_pressure_disturbance(
                   pressure, x_coords, block_x, effect_factor)

    vel_mag, x_vel, y_vel, z_vel = apply_blockage_velocity_disturbance(
                   vel_mag, x_vel, y_vel, z_vel,
                   x_coords, block_x, effect_factor)

    tke, tdr   = apply_blockage_turbulence_disturbance(
                   tke, tdr, x_coords, block_x, effect_factor)

    tailings   = apply_blockage_tailings_disturbance(
                   tailings, x_coords, block_x, effect_factor)

    wall_shear = apply_blockage_wall_shear_disturbance(
                   wall_shear, x_coords, block_x, effect_factor)

    # Clip to physical bounds
    pressure   = np.clip(pressure,    0,     45000)
    vel_mag    = np.clip(vel_mag,     0,     3.5)
    x_vel      = np.clip(x_vel,       0,     3.5)
    tke        = np.clip(tke,         0.023, 0.05)
    tdr        = np.clip(tdr,         0.07,  90.0)
    wall_shear = np.clip(wall_shear,  0,     55.0)
    tailings   = np.clip(tailings,    0.163, 0.508)

    p_coeff    = pressure * (66166 / 40473)

    df = pd.DataFrame({
        'nodenumber':           node_numbers,
        'x-coordinate':         x_coords,
        'y-coordinate':         y_coords,
        'z-coordinate':         z_coords,
        'pressure':             pressure,
        'pressure-coefficient': p_coeff,
        'density':              density,
        'velocity-magnitude':   vel_mag,
        'x-velocity':           x_vel,
        'y-velocity':           y_vel,
        'z-velocity':           z_vel,
        'turb-kinetic-energy':  tke,
        'turb-diss-rate':       tdr,
        'wall-shear':           wall_shear,
        'tailings-vof':         tailings,
        'timestep':             timestep,
        'label':                label_blockage,
    })

    return df

In [68]:
# Generate blockage cases
def generate_blockage_case(block_x, severity_name,
                            effect_factor, save=True):
    node_numbers, x_coords, y_coords, z_coords = \
        generate_node_coordinates()

    all_timesteps = []

    for t in range(1, timesteps + 1):
        df_t = generate_blockage_timestep(
            timestep      = t,
            block_x       = block_x,
            effect_factor = effect_factor,
            node_numbers  = node_numbers,
            x_coords      = x_coords,
            y_coords      = y_coords,
            z_coords      = z_coords,
        )
        all_timesteps.append(df_t)

        if t % 100 == 0:
            print(f"  Timestep {t}/{timesteps} done")

    df_case = pd.concat(all_timesteps, ignore_index=True)

    if save:
        loc_label = int(block_x)
        filename  = f"blockage_{severity_name}_loc{loc_label}m.csv"
        filepath  = Output_dir/ filename
        df_case.to_csv(filepath, index=False)
        print(f"Saved: {filepath}  shape={df_case.shape}")


    return df_case

In [69]:
print("=" * 60)
print("GENERATING ALL BLOCKAGE SCENARIOS")
print("=" * 60)

blockage_dataframes = {}

for loc_x in blockage_locations:
    for severity_name, effect_factor in blockage_severities.items():

        case_key = f"blockage_{severity_name}_{int(loc_x)}m"

        print(f"\nGenerating: {case_key}")
        print(f"  Location  : x = {loc_x} m")
        print(f"  Severity  : {severity_name}%")
        print(f"  Effect    : {effect_factor}")

        df_case = generate_blockage_case(
            block_x       = loc_x,
            severity_name = severity_name,
            effect_factor = effect_factor,
            save          = True,
        )

        blockage_dataframes[case_key] = df_case

print("\n" + "=" * 60)
print("ALL BLOCKAGE CASES GENERATED")
print(f"Total cases: {len(blockage_dataframes)}")
print("=" * 60)

GENERATING ALL BLOCKAGE SCENARIOS

Generating: blockage_25_8m
  Location  : x = 8.0 m
  Severity  : 25%
  Effect    : 0.25
  Timestep 100/700 done
  Timestep 200/700 done
  Timestep 300/700 done
  Timestep 400/700 done
  Timestep 500/700 done
  Timestep 600/700 done
  Timestep 700/700 done
Saved: /home/darlenewendie/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/synthetic/blockage_25_loc8m.csv  shape=(102200, 17)

Generating: blockage_50_8m
  Location  : x = 8.0 m
  Severity  : 50%
  Effect    : 0.55
  Timestep 100/700 done
  Timestep 200/700 done
  Timestep 300/700 done
  Timestep 400/700 done
  Timestep 500/700 done
  Timestep 600/700 done
  Timestep 700/700 done
Saved: /home/darlenewendie/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/synthetic/blockage_50_loc8m.csv  shape=(102200, 17)

Generating: blockage_75_8m
  Location  : x = 8.0 m
  Severity  : 75%
  Effect    : 0.9
  Timestep 100/700 done
  Timestep 200/700 done
  Timestep 300/700 done
  Timestep 400/700 don

In [70]:
# Combine all 15 blockage cases
print("Combining all blockage cases...")
df_all_blockages = pd.concat(
    list(blockage_dataframes.values()),
    ignore_index=True
)
print(f"Combined shape: {df_all_blockages.shape}")

# Save combined

# Save
combined_path = Path(os.path.join(os.path.dirname(os.getcwd()), "data", "raw"))
blockage_combined_path = os.path.join(combined_path, "all_blockages_combined.csv")
df_all_blockages.to_csv(blockage_combined_path, index=False)
print(f"Saved: {blockage_combined_path}")

# Sample 102,200 to match normal
df_blockage_sampled = df_all_blockages.sample(
    n=102_200,
    random_state=42,
    replace=False
).reset_index(drop=True)

print(f"\nSampled shape  : {df_blockage_sampled.shape}")
print(f"Label = 2 only : {(df_blockage_sampled['label'] == 2).all()}")

# Save
blockage_sample_data_path = Path(os.path.join(os.path.dirname(os.getcwd()), "data", "raw"))
blockage_sampled_path = os.path.join(blockage_sample_data_path, "blockage_dataset_sampled.csv")
df_blockage_sampled.to_csv(blockage_sampled_path , index=False)
print(f"Saved: {blockage_sampled_path}")


Combining all blockage cases...
Combined shape: (1533000, 17)
Saved: /home/darlenewendie/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/raw

Sampled shape  : (102200, 17)
Label = 2 only : True
Saved: /home/darlenewendie/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/raw/blockage_dataset_sampled.csv


In [71]:
print("VALIDATION — Blockage physics check")
print("=" * 60)

sample_case = blockage_dataframes['blockage_75_25m']
ts_sample   = sample_case[sample_case['timestep'] == 350].copy()
ts_sample   = ts_sample.sort_values('x-coordinate')

near_block     = ts_sample[abs(ts_sample['x-coordinate'] - 25) < 1.0]
far_upstream   = ts_sample[ts_sample['x-coordinate'] < 10]
far_downstream = ts_sample[ts_sample['x-coordinate'] > 40]

print("\nPressure comparison:")
print(f"  Far upstream   : {far_upstream['pressure'].mean():.1f} Pa")
print(f"  Near blockage  : {near_block['pressure'].mean():.1f} Pa")
print(f"  Far downstream : {far_downstream['pressure'].mean():.1f} Pa")

print("\nVelocity comparison:")
print(f"  Far upstream   : {far_upstream['velocity-magnitude'].mean():.4f} m/s")
print(f"  Near blockage  : {near_block['velocity-magnitude'].mean():.4f} m/s")
print(f"  Far downstream : {far_downstream['velocity-magnitude'].mean():.4f} m/s")

print("\nTailings VOF comparison:")
print(f"  Far upstream   : {far_upstream['tailings-vof'].mean():.4f}")
print(f"  Near blockage  : {near_block['tailings-vof'].mean():.4f}")
print(f"  Far downstream : {far_downstream['tailings-vof'].mean():.4f}")

print("\nExpected physics:")
print("  Pressure  : upstream HIGHEST → blockage spike → downstream LOWEST")
print("  Velocity  : near blockage LOWEST")
print("  Tailings  : upstream elevated → near blockage HIGHEST → downstream LOWEST")

VALIDATION — Blockage physics check

Pressure comparison:
  Far upstream   : 36004.6 Pa
  Near blockage  : 19314.4 Pa
  Far downstream : 3762.6 Pa

Velocity comparison:
  Far upstream   : 2.4740 m/s
  Near blockage  : 2.1735 m/s
  Far downstream : 2.4703 m/s

Tailings VOF comparison:
  Far upstream   : 0.4003
  Near blockage  : 0.4087
  Far downstream : 0.4016

Expected physics:
  Pressure  : upstream HIGHEST → blockage spike → downstream LOWEST
  Velocity  : near blockage LOWEST
  Tailings  : upstream elevated → near blockage HIGHEST → downstream LOWEST


In [72]:
print("Blockage sample distribution check:")
print(f"\nX-coordinate coverage:")
print(df_blockage_sampled['x-coordinate'].describe()[['min','max','mean']])

print(f"\nTimestep coverage:")
print(df_blockage_sampled['timestep'].describe()[['min','max','mean']])

# Check physics still visible in sample
near_any_block = df_blockage_sampled[
    df_blockage_sampled['x-coordinate'].apply(
        lambda x: min(abs(x - loc) for loc in [8,16,25,34,42]) < 1.0
    )
]
far_from_blocks = df_blockage_sampled[
    df_blockage_sampled['x-coordinate'].apply(
        lambda x: min(abs(x - loc) for loc in [8,16,25,34,42]) > 5.0
    )
]

print(f"\nNear blockage — mean pressure : {near_any_block['pressure'].mean():.1f} Pa")
print(f"Far from block — mean pressure : {far_from_blocks['pressure'].mean():.1f} Pa")
print(f"\nNear blockage — mean velocity : {near_any_block['velocity-magnitude'].mean():.4f} m/s")
print(f"Far from block — mean velocity : {far_from_blocks['velocity-magnitude'].mean():.4f} m/s")
print(f"\nNear blockage — mean tailings : {near_any_block['tailings-vof'].mean():.4f}")
print(f"Far from block — mean tailings : {far_from_blocks['tailings-vof'].mean():.4f}")

print("\nExpected:")
print("  Near blockage pressure  > far pressure  (back pressure)")
print("  Near blockage velocity  < far velocity  (flow restricted)")
print("  Near blockage tailings  > far tailings  (solids accumulating)")

Blockage sample distribution check:

X-coordinate coverage:
min      0.000000
max     50.000000
mean    24.968689
Name: x-coordinate, dtype: float64

Timestep coverage:
min       1.000000
max     700.000000
mean    350.921957
Name: timestep, dtype: float64

Near blockage — mean pressure : 20028.9 Pa
Far from block — mean pressure : 20126.8 Pa

Near blockage — mean velocity : 2.4369 m/s
Far from block — mean velocity : 2.4788 m/s

Near blockage — mean tailings : 0.4003
Far from block — mean tailings : 0.4000

Expected:
  Near blockage pressure  > far pressure  (back pressure)
  Near blockage velocity  < far velocity  (flow restricted)
  Near blockage tailings  > far tailings  (solids accumulating)
